# Proyek Akhir: Menyelesaikan Permasalahan Perusahaan Edutech

- Nama: Rizky Maulana Ayub
- Email: rizkymaulanaayub@gmail.com
- Id Dicoding: rizbuy

## Persiapan

### Menyiapkan library yang dibutuhkan

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_squared_error
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import joblib

### Menyiapkan data yang akan digunakan


## Data Understanding

In [ ]:
df = pd.read_csv('employee_data.csv')

In [ ]:
df_dashboard = df.copy()

In [ ]:
df_dashboard.describe().T

In [ ]:
msno.matrix(df_dashboard)
print(df_dashboard.isna().sum())

In [ ]:
df_dashboard.duplicated().sum()

In [ ]:
df_dashboard.nunique()

In [ ]:
numerical_cols = df_dashboard.select_dtypes(include=['number'])
print(f'Numerical columns: {numerical_cols}')

In [ ]:
numerical_cols.hist(bins=30, figsize=(15, 10), grid=True)
plt.suptitle('Distribusi Kolom Numerik', fontsize=20)
plt.tight_layout()
plt.show()

In [ ]:
cat_cols = df_dashboard.select_dtypes(include=['object','str']).columns
print(f'Categorical columns: {cat_cols}')
print(cat_cols)

In [ ]:
df_corr = df_dashboard.select_dtypes(include=['number']).corr()
plt.figure(figsize=(20, 20))
sns.heatmap(df_corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Numerical Features')
plt.show()

In [ ]:
df_dashboard.info()

In [ ]:
plt.figure(figsize=(12, 8))
plot = sns.barplot(x='MaritalStatus', y='MonthlyIncome', hue='OverTime', data=df_dashboard, estimator=np.mean, errorbar=None)

for p in plot.patches:
    height = p.get_height()
    plot.annotate(f'{height:.2f}', (p.get_x() + p.get_width() / 2., height), ha='center', va='bottom')

plt.title('Rata-rata Monthly Income berdasarkan Marital Status dan OverTime', fontsize=14)
plt.xlabel('Marital Status', fontsize=12)
plt.ylabel('Average Monthly Income', fontsize=12)
plt.legend(title='OverTime', loc='best')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='Attrition', y='MonthlyIncome', data=df_dashboard, errorbar=None, estimator=np.mean)
plt.title('Hubungan Pendapatan Bulanan dengan Attrition')
plt.ylabel('Monthly Income')
plt.show()

In [ ]:
businees_travel_attrition = pd.crosstab(df_dashboard['BusinessTravel'], df_dashboard['Attrition'])
plot = businees_travel_attrition.plot(kind='bar', stacked=False, figsize=(10, 6))
for p in plot.patches:
    height = p.get_height()
    plot.annotate(f'{height:.0f}', (p.get_x() + p.get_width() / 2., height), ha='center', va='bottom')
plt.title('Persentase Attrition Berdasarkan Business Travel')
plt.ylabel('Jumlah Karyawan')
plt.xticks(rotation=0)
plt.legend(title='Attrition', loc='best')
plt.show()

In [ ]:
# Cross-tabulation antara OverTime dan Attrition
overtime_attrition = pd.crosstab(df_dashboard['OverTime'], df_dashboard['Attrition'], normalize='index') * 100
plot = overtime_attrition.plot(kind='bar', stacked=False, figsize=(8, 5))
for p in plot.patches:
    height = p.get_height()
    plot.annotate(f'{height:.2f}%', (p.get_x() + p.get_width() / 2., height), ha='center', va='bottom')
plt.title('Persentase Attrition Berdasarkan Lembur (OverTime)')
plt.ylabel('Persentase (%)')
plt.xticks(rotation=0)
plt.legend(title='Attrition', loc='best')
plt.show()

## Data Preparation / Preprocessing

In [ ]:
df_dashboard.dropna(inplace=True)
df_dashboard.info()

In [ ]:
df_dashboard.drop(columns=['Over18', 'EmployeeCount', 'StandardHours'], inplace=True)

In [ ]:
# Mapping Education Level
education_mapping = {
    1: 'Below College',
    2: 'College',
    3: 'Bachelor',
    4: 'Master',
    5: 'Doctor'
}
df_dashboard['Education'] = df_dashboard['Education'].map(education_mapping)
print("Education Level Mapping:")
print(df_dashboard['Education'].value_counts())

In [ ]:
satisfaction_mapping = {
    1: 'Low',
    2: 'Medium',
    3: 'High',
    4: 'Very High'
}
df_dashboard['JobSatisfaction'] = df_dashboard['JobSatisfaction'].map(satisfaction_mapping)
df_dashboard['EnvironmentSatisfaction'] = df_dashboard['EnvironmentSatisfaction'].map(satisfaction_mapping)
df_dashboard['RelationshipSatisfaction'] = df_dashboard['RelationshipSatisfaction'].map(satisfaction_mapping)
df_dashboard['JobInvolvement'] = df_dashboard['JobInvolvement'].map(satisfaction_mapping)
print("Job Satisfaction Mapping:")
print(df_dashboard['JobSatisfaction'].value_counts())
print("\nEnvironment Satisfaction Mapping:")
print(df_dashboard['EnvironmentSatisfaction'].value_counts())
print("\nRelationship Satisfaction Mapping:")
print(df_dashboard['RelationshipSatisfaction'].value_counts())
print("\nJob Involvement Mapping:")
print(df_dashboard['JobInvolvement'].value_counts())

In [ ]:
rating_mapping = {
    1: 'Low',
    2: 'Good',
    3: 'Excellent',
    4: 'Outstanding'
}
df_dashboard['PerformanceRating'] = df_dashboard['PerformanceRating'].map(rating_mapping)
print("Performance Rating Mapping:")
print(df_dashboard['PerformanceRating'].value_counts())
df_dashboard['WorkLifeBalance'] = df_dashboard['WorkLifeBalance'].map(rating_mapping)
print("Work-Life Balance Mapping:")
print(df_dashboard['WorkLifeBalance'].value_counts())

In [ ]:
df_dashboard['Attrition'] = df_dashboard['Attrition'].apply(lambda x: 'Yes' if x == 1 else 'No')

In [ ]:
pd.options.display.max_columns = 500
df_dashboard.head()

In [ ]:
df_dashboard.info()

In [ ]:
df_dashboard.describe(include='all').T

In [ ]:
df_jobrole = df_dashboard.groupby(['Gender', 'JobRole']).agg({'Age':['min', 'max'], 'MonthlyIncome':['min', 'max']})
print(df_jobrole)

In [ ]:
df_department = df_dashboard.groupby(['Department', 'MaritalStatus']).agg({'Age':['min', 'max'], 'MonthlyIncome':['min', 'max', 'mean']})
print(df_department)

In [ ]:
df_education = df_dashboard.groupby(['Education', 'EducationField']).agg({'Age':['min', 'max'], 'MonthlyIncome':['min', 'max', 'mean']})
print(df_education)

In [ ]:
df_income = df_dashboard.groupby(['OverTime', 'MaritalStatus']).agg({'MonthlyIncome':['min', 'max', 'mean']})
print(df_income)

In [ ]:
# Hitung Attrition Rate
attrition_count = (df_dashboard['Attrition'] == 'Yes').sum()
total_employees = len(df_dashboard)
attrition_rate = (attrition_count / total_employees) * 100
# Hitung Attrition Rate
attrition_count = (df_dashboard['Attrition'] == 'Yes').sum()
total_employees = len(df_dashboard)
attrition_rate = (attrition_count / total_employees) * 100

print(f'Total Karyawan: {total_employees}')
print(f'Jumlah Attrition: {attrition_count}')
print(f'Persentase Attrition: {attrition_rate}')

# Attrition Breakdown
attrition_breakdown = df_dashboard['Attrition'].value_counts()
print('\nBreakdown Attrition:')
print(attrition_breakdown)
print(f'\nPersentase Attrition:')
print(df_dashboard['Attrition'].value_counts(normalize=True) * 100)

# Visualisasi Attrition Rate
plt.figure(figsize=(10, 6))
attrition_labels = ['Tetap Bekerja (No)', 'Keluar (Yes)']
no_count = (df_dashboard['Attrition'] == 'No').sum()
yes_count = (df_dashboard['Attrition'] == 'Yes').sum()
attrition_values = [no_count, yes_count]
colors = ['#2ecc71', '#e74c3c']
plt.pie(attrition_values, labels=attrition_labels, autopct='%1.2f%%', colors=colors, startangle=90)
plt.title(f'Attrition Rate: {attrition_rate:.2f}%', fontsize=14, fontweight='bold')
plt.show()


In [ ]:
df_dashboard.to_csv('employee_data_cleaned.csv', index=False)
print('Data telah dibersihkan dan disimpan sebagai employee_data_cleaned.csv')

# Modeling

In [ ]:
df_dashboard.tail()

In [ ]:
pd.options.display.max_columns=500
df_dashboard.describe(include='str').T

In [ ]:
df_model = df_dashboard.copy()
df_model.head()

In [ ]:
# 1. Definisikan urutan kategori sesuai deskripsi dataset Anda
education_order = ['Below College', 'College', 'Bachelor', 'Master', 'Doctor']
satisfaction_order = ['Low', 'Medium', 'High', 'Very High'] # Untuk EnvSat, JobSat, dan RelationshipSat
involvement_order = ['Low', 'Medium', 'High', 'Very High']
performance_order = ['Low', 'Good', 'Excellent', 'Outstanding']
worklife_order = ['Low', 'Good', 'Excellent', 'Outstanding']
# Catatan: Untuk BusinessTravel, biasanya urutannya adalah:
travel_order = ['Non-Travel', 'Travel_Rarely', 'Travel_Frequently']

# 2. Inisialisasi OrdinalEncoder dengan urutan di atas
ordinal_enc = OrdinalEncoder(categories=[
    education_order,
    satisfaction_order, # EnvironmentSatisfaction
    involvement_order,
    satisfaction_order, # JobSatisfaction
    performance_order,
    satisfaction_order, # RelationshipSatisfaction
    worklife_order,
    travel_order
])

# 3. Tentukan kolom yang akan di-encode secara ordinal
ordinal_cols = [
    'Education',
    'EnvironmentSatisfaction',
    'JobInvolvement',
    'JobSatisfaction',
    'PerformanceRating',
    'RelationshipSatisfaction',
    'WorkLifeBalance',
    'BusinessTravel'
]

# 4. Terapkan pada DataFrame
df_model[ordinal_cols] = ordinal_enc.fit_transform(df_model[ordinal_cols])

In [ ]:
df_model['Attrition'] = df_model['Attrition'].apply(lambda x: 1 if x == 'Yes' else 0)
df_model['OverTime'] = df_model['OverTime'].apply(lambda x: 1 if x == 'Yes' else 0)
df_model['Gender'] = df_model['Gender'].apply(lambda x: 1 if x == 'Male' else 0)

In [ ]:
nominal_cols = [
    'Department',
    'MaritalStatus',
    'JobRole',
    'EducationField',
]
df_model = pd.get_dummies(df_model, columns=nominal_cols, drop_first=True)

In [ ]:
df_model.tail()

In [ ]:
X = df_model.drop(columns=['EmployeeId', 'Attrition'])
y = df_model['Attrition']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
print(f'akurasi:', {accuracy})
print(f'Precision:', {precision})
print(f'Recall:', {recall})
print(f'F1:', {f1})

In [ ]:
train_acc = rf_model.score(X_train, y_train)
test_acc = rf_model.score(X_test, y_test)
print(f'Train Accuracy:', {train_acc})
print(f'Test Accuracy:', {test_acc})

In [ ]:
param_grid = {
    'n_estimators' : [250, 300, 350],
    'criterion' : ['gini', 'entropy'],
    'max_features' : ['sqrt', 'log2'],
    'min_samples_split' : [2, 5, 10, 20],
    'min_samples_leaf' : [2, 5, 10, 20],
    'max_depth' : [None, 10, 20]
}
grid_search = GridSearchCV(estimator=rf_model,
                           param_grid=param_grid,
                           scoring='accuracy',
                           cv=3,
                           n_jobs=-1,
                           verbose=1
)
grid_search.fit(X_train, y_train)
print('Best params:', grid_search.best_params_)
print('Best Accuracy:', grid_search.best_score_)
best_model = grid_search.best_estimator_
print('test accuracy', best_model.score(X_test, y_test))

## Evaluation

In [ ]:
y_pred = best_model.predict(X_test)
grid_mse = mean_squared_error(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
print(f"Akurasi: {accuracy}")
print(f"Mean Squared Error: {grid_mse}")
print(f"Presisi: {precision}")
print(f"Recall: {recall}")
print(f"F1-Score: {f1}")

In [ ]:
# 1. Ambil feature importance dari model terbaik
importances = best_model.feature_importances_
feature_names = X.columns

feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance_df['Feature'].head(10), feature_importance_df['Importance'].head(10), color='skyblue')
plt.xlabel('Importance Score')
plt.title('Top 10 Faktor Penyebab Attrition (Random Forest)')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.kdeplot(data=df_model, x='MonthlyIncome', hue='Attrition', fill=True, color='skyblue')
plt.title('Distribusi Monthly Income berdasarkan Attrition')
plt.show()

In [ ]:
y_pred = best_model.predict(X_test)

print('Classification Report:')
print(classification_report(y_test, y_pred))

matriks = confusion_matrix(y_test, y_pred)
display = ConfusionMatrixDisplay(confusion_matrix=matriks, display_labels=['Stay', 'Attrition'])

plt.figure(figsize=(10, 6))
display.plot(cmap='Blues')
plt.title('Prediksi Confusion Matrix')
plt.show()

In [ ]:
# 1. Simpan Model Terbaik
joblib.dump(best_model, 'best_model.pkl')

# 2. Simpan Encoder Ordinal (sangat penting agar mapping teks -> angka tidak tertukar)
joblib.dump(ordinal_enc, 'ordinal_encoder.pkl')

print("Model dan Encoder berhasil disimpan!")

In [ ]:
df_dashboard.tail(20).to_csv('test_data_raw.csv', index=False)